# 29B Actions, approvals and the audit ledger
Reading and drafting are free; anything that leaves the firm, or changes what a client relies on, needs Priya's approval of the exact payload. Standing rules live in the server.

In [1]:
import sys, json, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "creditwatch").exists())
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd, yaml
from creditwatch import (approvals, audit, catalog, channel, clock, econ, evals, heartbeat, llm, loop,
                         memory, panel, policy, signals, skills, standin, workspace)
P = yaml.safe_load(open(ROOT / "params.yaml"))
FX = ROOT / P["data"]["fixture_dir"]
USERS = json.load(open(FX / "users.json"))
MANDATES = json.load(open(ROOT / P["data"]["mandates"]))
REG = json.load(open(ROOT / P["data"]["registry"]))
PANEL = panel.load_panel(ROOT / P["data"]["panel"])
M = P["heartbeat"]["materiality"]
NOW = clock.clock()                       # LANTERN_CLOCK, never the system clock
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 20); pd.set_option("display.max_colwidth", 60)
print("fixture mode:", P["data"]["use_fixture"], "| clock:", NOW.isoformat())

fixture mode: True | clock: 2026-10-15T06:00:00-04:00


In [2]:
others = policy.confidential_terms(MANDATES, "mandate_a")
base = {"draft_id": "d-001", "scope": "mandate_a", "to": ["cio@harborstate.example"], "subject": "Juliett after its 10-Q",
        "body": "Juliett marked Crestline Roofing down 7.5 points, to 90.0, in its June 10-Q (0009000014-26-000141).", "attachments": []}
cases = {"the reply Priya would approve": {},
         "answers the question about Northgate": {"body": base["body"] + " Northgate has been adding to Hotel."},
         "obfuscated account code of the other client": {"body": base["body"] + " Ref NG\u200b-PC 9902."},
         "names a restricted issuer": {"body": base["body"] + " Ironwood Data Systems is refinancing."},
         "writes to the borrower": {"to": ["cfo@crestlineroofing.example"]},
         "copies an unknown address": {"to": ["cio@harborstate.example", "me@gmail.example"]}}
for name, change in cases.items():
    print(f"{name:45s} -> {policy.policy_check({**base, **change}, MANDATES['mandate_a'], REG, others) or 'allowed'}")

the reply Priya would approve                 -> allowed
answers the question about Northgate          -> R-OTHER-MANDATE
obfuscated account code of the other client   -> R-OTHER-MANDATE
names a restricted issuer                     -> R-RESTRICTED
writes to the borrower                        -> R-BORROWER
copies an unknown address                     -> R-NOT-CONTACT


In [3]:
t0 = NOW.timestamp()
tok = approvals.issue_token(base, "priya", now=t0, ttl_min=P["approvals"]["ttl_minutes"])
print("hash:", approvals.payload_hash(base)[:16], "| token:", tok[:24] + "...")
print("same payload        ->", approvals.verify_token(base, tok, "priya", t0 + 60))
print("recipient added     ->", approvals.verify_token({**base, "to": base["to"] + ["credit.pm@harborstate.example"]}, tok, "priya", t0 + 60))
print("31 minutes later    ->", approvals.verify_token(base, tok, "priya", t0 + 31 * 60))
print("presented for Tom   ->", approvals.verify_token(base, tok, "tom", t0 + 60))

hash: f178d50564ccfc75 | token: priya.1792060200.afc26ba...
same payload        -> ok
recipient added     -> payload_changed
31 minutes later    -> expired
presented for Tom   -> wrong_approver


In [5]:
# watchlist_write is an INTERNAL_WRITE with an approval override: clients rely on the watchlist.
entry = {"draft_id": "w-001", "scope": "mandate_b", "bdc": "FIXTURE HOTEL SENIOR LENDING",
         "reason": "Brightwater down 5.5 points; Oakmont PIK up from 1.00% to 3.00% (0009000012-26-000121)"}
print("policy for watchlist_write:", catalog.policy_for("watchlist_write"))
wtok = approvals.issue_token(entry, "priya", now=t0)
print("approved entry ->", approvals.verify_token(entry, wtok, "priya", t0 + 5),
      "| reason edited afterwards ->", approvals.verify_token({**entry, "reason": "fine"}, wtok, "priya", t0 + 5))

policy for watchlist_write: approve
approved entry -> ok | reason edited afterwards -> payload_changed


In [ ]:
# Each notebook run gets its own persistent audit file (no overwritten history).
from uuid import uuid4
run_id = "lab29-demo-" + uuid4().hex
audit_dir = ROOT / "logs" / "audit"
audit_dir.mkdir(parents=True, exist_ok=True)
audit_path = audit_dir / f"{run_id}.jsonl"
led = audit.Ledger(path=audit_path, clock=clock.clock)
led.record("tool_call", tool="draft_email", scope="mandate_a", decision="executed", payload_hash=approvals.payload_hash(base))
led.record("approval_requested", tool="send_email", payload_hash=approvals.payload_hash(base))
led.record("approval_granted", approver="priya", payload_hash=approvals.payload_hash(base))
swapped = {**base, "to": base["to"] + ["credit.pm@harborstate.example"]}
led.record("policy_block", tool="send_email", rule_id="R-TOKEN", decision="blocked",
           reason=approvals.verify_token(swapped, tok, "priya", t0 + 60), payload_hash=approvals.payload_hash(swapped))
print(pd.DataFrame(led.events).fillna("")[["event_id", "ts", "kind", "tool", "decision", "rule_id", "reason"]])
print("\nThe ledger stores hashes and ids, never the email body.")
print("Saved demonstration audit log:", audit_path)
print("These are simulated approval events; no email was sent.")


   event_id                         ts                kind         tool  decision  rule_id           reason
0  ev-00001  2026-10-15T06:00:00-04:00           tool_call  draft_email  executed                          
1  ev-00002  2026-10-15T06:00:00-04:00  approval_requested   send_email                                    
2  ev-00003  2026-10-15T06:00:00-04:00    approval_granted                                                 
3  ev-00004  2026-10-15T06:00:00-04:00        policy_block   send_email   blocked  R-TOKEN  payload_changed

The ledger stores hashes and ids, never the email body.
Saved demonstration audit log: /Users/sailokeshreddynandavarapu/Documents/Sem_5_Fall_26/Lab_Activity/Lab3b/DAMG-7245-Project-Lantern-Case-Study-Lab3B-Team-2/cs3b/logs/audit/lab29-demo-82207db9d708412f870c896f22c85c94.jsonl
These are simulated approval events; no email was sent.


### Review saved demonstration logs
Cell 5 now saves one JSONL file per run under `logs/audit/`. Each line is an audit event. The blocked send records `reason: payload_changed`. These are notebook demonstration events, not evidence of a real human approval or email delivery. Email bodies and approval tokens are not stored.
The following cell reads these files from disk and works after a kernel restart.

In [7]:
# Read saved logs from disk, even after restarting the notebook kernel.
import json
from pathlib import Path
import pandas as pd
root = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "src/creditwatch").is_dir())
blocked = []
for log_file in sorted((root / "logs/audit").glob("lab29-demo-*.jsonl")):
    for line in log_file.read_text().splitlines():
        if line.strip():
            event = json.loads(line)
            if event.get("kind") == "policy_block":
                blocked.append({"log_file": log_file.name, **event})
if blocked:
    print(pd.DataFrame(blocked)[["log_file", "ts", "tool", "decision", "rule_id", "reason"]].to_string(index=False))
else:
    print("No saved Lab 29 blocked-send events yet. Run code cell 5 first.")


                                         log_file                        ts       tool decision rule_id          reason
lab29-demo-82207db9d708412f870c896f22c85c94.jsonl 2026-10-15T06:00:00-04:00 send_email  blocked R-TOKEN payload_changed


**Your turn.** Build `draft_email`, `request_approval`, `send_email` and `watchlist_write` in your server, with the policy engine reading action classes from `params.yaml`. Write tests for every blocked case in Part 6, and prove a correctly approved send reaches the mock outbox: a system that blocks everything fails.

## Your turn — configured actions, approvals and saved audit

- The existing four action tools now use the action classes and policies in `params.yaml`. Unsafe class changes or removal of mandatory approval fail at startup.
- The server enforces content/recipient rules and checks the exact approved payload, principal and expiry before mock send or watchlist write.
- Approval requests are not grants. A draft changed after the request requires a new request; a changed approved draft requires new approval.
- Authenticated MCP sessions persist redacted audit events. Blocked actions record reviewable reasons such as `payload_changed` and `expired`; tokens and email bodies are not copied into the log.
- Acceptance tests cover notebook blocked cases, changed fields, wrong/expired/missing approvals, stale requests, configuration enforcement and successful send/watchlist paths.
- A labelled fixture exercise saved a blocked-recipient-change event, then obtained simulated fresh approval and wrote the email to the local mock outbox. A repeated send returned `already_sent`.

The exercise uses simulated human grants and makes no external email delivery. Its watchlist changes affect a scratch workspace only. For interactive approval use `python -m services.mcp.approve REQUEST_ID --mandate mandate_a`.


In [ ]:
from pathlib import Path
import json, sys, subprocess
root = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "src/creditwatch").is_dir())
sys.path[:0] = [str(root), str(root / "src")]
# Read the saved server-backed fixture demonstration. Run
# `python -m services.mcp.demo_actions` to generate another one.
reports = sorted((root / "reports/actions").glob("lab29-actions-*.json"),
                 key=lambda p: p.stat().st_mtime)
if not reports:
    from services.mcp.demo_actions import run
    run(root)
    reports = sorted((root / "reports/actions").glob("lab29-actions-*.json"))
report = json.loads(reports[-1].read_text())
print(json.dumps(report, indent=2))
print("\nSAVED BLOCKED EVENTS:")
for line in (root / report["audit_log"]).read_text().splitlines():
    event = json.loads(line)
    if event["kind"] == "policy_block":
        print({k: event[k] for k in ("tool", "decision", "rule_id", "reason")})
print("Local mock-outbox file exists:", (root / report["outbox_file"]).exists())
result = subprocess.run([sys.executable, "-m", "pytest", "tests/test_action_controls.py",
                         "-q", "-p", "no:cacheprovider"], cwd=root, capture_output=True, text=True)
print(result.stdout)
if result.returncode:
    print(result.stderr)
result.check_returncode()


{
  "approved_send": {
    "outbox_message_id": "msg-af619290b13712c7aefca267b04e234b6498c1b1fbd21cb2a8f36a06e9427bf4",
    "status": "sent"
  },
  "audit_log": "logs/audit/lab29-actions-e00173a3af694a16b804538b5f6059c1.jsonl",
  "blocked_reason": "payload_changed",
  "mode": "fixture exercise with simulated human approvals; no external email",
  "outbox_file": "outputs/outbox/msg-af619290b13712c7aefca267b04e234b6498c1b1fbd21cb2a8f36a06e9427bf4.json",
  "retry_status": "already_sent",
  "run_id": "lab29-actions-e00173a3af694a16b804538b5f6059c1",
  "scratch_workspace": ".cache/lab29-actions-e00173a3af694a16b804538b5f6059c1/agent",
  "watchlist_write": {
    "status": "written",
    "watchlist_entry_id": "watch-4cbbc834dbe2881e001c43c32a5613731ac8acfe2a9a7385ab50abc4df43c15b"
  }
}

SAVED BLOCKED EVENTS:
{'tool': 'send_email', 'decision': 'blocked', 'rule_id': 'R-TOKEN', 'reason': 'payload_changed'}
Local mock-outbox file exists: True
.............................                        